# AI-related skill mix across occupation categories

This notebook selects **10 occupations from each of 11 broad categories** in the supplied CSV, then plots a horizontal 100% stacked bar for each occupation. The 3 segments show the share of **classified skills** that match the three lists.

**Selection rule:** within each SOC occupation family, the notebook picks the 10 occupations with the most skills matching the lists. This is a useful exploratory sample, not a random or representative sample. The percentages are not direct estimates of the percentage of a job that AI can replace.

Place `lc_occupation_skills.csv` in the same folder as this notebook before running it.

In [ ]:
# Skill category distribution across 110 occupations
# Dataset: 2023, Full year, US. Selects 10 occupations per broad category with the most
# matches to the three skill lists, so each selected bar has enough classified skills to show.

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# If the CSV is in the same folder as this notebook, this path is sufficient.
df = pd.read_csv("lc_occupation_skills.csv")

df = df[
    (df["year"] == 2023) &
    (df["period"] == "Full year") &
    (df["geography"] == "US")
].copy()


In [ ]:
# Use the exact skill-name lists from the previous classification.
human_centric = ['Nursing', 'Registered Nurse (RN)', 'Certified Nursing Assistant (CNA)', 'Patient Treatment', 'Patient Assistance', 'Patient Safety', 'Bedside Care', 'Home Health Care', 'Surgery', 'Emergency Medicine', 'Trauma Care', 'Physical Therapy Evaluation', 'Medication Administration', 'Phlebotomy', 'Midwifery License', 'Psychotherapy', 'Crisis Intervention', 'Licensed Clinical Social Worker (LCSW)', 'Patient Education And Counseling', 'Working With Children', 'Special Education', 'Classroom Management', 'Teaching', 'Mentorship', 'Coaching', 'Emotional Intelligence', 'Compassion', 'Rapport Building', 'Conflict Handling', 'HVAC Repair And Maintenance', 'Plumbing', 'Electrical Wiring', 'Carpentry', 'Welding', 'Aircraft Maintenance', 'Machining', 'Repairing (Manual/Mechanical)', 'Landscaping', 'Cooking', 'Housekeeping', 'Equipment Inspection', 'Patient Positioning', 'Dental Care', 'Animal Care', 'First Aid', 'Emergency Response', 'Nursing Care']

ai_centric = ['Data Entry', 'Data Processing', 'Data Recording', 'Data Compilation', 'Data Preprocessing', 'Data Conversion', 'Data Documentation', 'Data Management', 'Typing', 'Filing', 'Mail Sorting', 'Sorting', 'Transcribing', 'Proofreading', 'Document Formatting', 'Memos', 'Technical Documentation', 'Medical History Documentation', 'Regulatory Documents', 'Legal Discovery', 'Language Translation', 'Administrative Support', 'Administrative Functions', 'Office Procedures', 'Office Administration', 'Calendar Management', 'Scheduling', 'Travel Arrangements', 'Reservations', 'Invoicing', 'Billing', 'Payroll Processing', 'Transaction Processing (Business)', 'Cash Handling', 'Customer Inquiries', 'Customer Support', 'Customer Complaint Resolution', 'Call Center Experience', 'Mail Forwarding', 'Package Sorting', 'Record Keeping', 'Data Collection', 'Billing Inquiries', 'Office Automation', 'Switchboard Operator', 'Multi-Line Phone Systems']

human_utilizing_ai_centric = ['Software Engineering', 'Software Development', 'Application Development', 'Python (Programming Language)', 'Java (Programming Language)', 'SQL (Programming Language)', 'C++ (Programming Language)', 'Debugging', 'Algorithm Design', 'Machine Learning', 'Artificial Intelligence', 'Deep Learning', 'Cyber Security', 'Cloud Computing', 'Amazon Web Services', 'Database Management Systems', 'Data Science', 'Data Analysis', 'Quantitative Data Analysis', 'Applied Statistics', 'Bayesian Inference', 'Causal Inference', 'Data-Driven Decision Making', 'Microsoft Excel', 'Power BI', 'Business Intelligence', 'Economic Modeling', 'Financial Analysis', 'Risk Management', 'Competitive Analysis', 'AutoCAD', 'Computer-Aided Design', 'SolidWorks (CAD)', '3D Modeling', 'Product Design', 'Prototyping', 'Engineering Design Process', 'Computational Fluid Dynamics', 'MATLAB', 'Robotics', 'Automation Systems Design', 'Project Management', 'Project Planning', 'Workflow Management', 'Business Workflow Analysis', 'Business Systems Analysis', 'Process Improvement', 'Business Process Improvement', 'Supply Chain Management', 'Supply Chain Planning', 'Strategic Leadership', 'Decision Making', 'Team Management', 'Change Leadership', 'Vendor Relationship Management', 'Marketing', 'Content Marketing', 'Content Development', 'Campaign Management', 'Creative Direction', 'Sales Prospecting', 'Sales', 'Customer Relationship Management (CRM) Software', 'Salesforce', 'Demand Generation', 'Prospect Research', 'Public Relations', 'Proposal Development', 'Technical Research', 'Human Resources Information System (HRIS)', 'Compensation Analysis', 'Benefits Administration', 'Instructional Design', 'Curriculum Development', 'Educational Assessment', 'Learning Management Systems', 'Data Visualization', 'Research', 'Writing', 'Report Writing', 'Dashboard', 'Information Synthesis', 'Policy Analysis']


In [ ]:
category_order = [
    "Human-centric",
    "AI-centric",
    "Human-utilizing-AI-centric",
]

skill_to_category = {}
for skill in human_centric:
    skill_to_category[skill] = "Human-centric"
for skill in ai_centric:
    skill_to_category[skill] = "AI-centric"
for skill in human_utilizing_ai_centric:
    skill_to_category[skill] = "Human-utilizing-AI-centric"

# Safety checks: no overlap and all labels are exact skill names from the CSV.
all_listed_skills = human_centric + ai_centric + human_utilizing_ai_centric
assert len(all_listed_skills) == len(set(all_listed_skills)), "The lists overlap."
dataset_skill_names = set(df["skill_name"].dropna().unique())
missing = [s for s in all_listed_skills if s not in dataset_skill_names]
assert not missing, f"Skills not found exactly in the dataset: {missing}"

df["category"] = df["skill_name"].map(skill_to_category)
classified = df.dropna(subset=["category"]).drop_duplicates(subset=["onet", "skill_name"])

# Count how many classified skills from each category are listed for each occupation.
counts = pd.crosstab(
    [classified["onet"], classified["onet_name"]],
    classified["category"]
).reindex(columns=category_order, fill_value=0)
counts["total_classified"] = counts.sum(axis=1)


In [ ]:
# Map broad occupation families to Standard Occupational Classification (SOC) codes.
# The marketing-related SOCs are assigned to Sales/Marketing/Customer Service to avoid duplicates.
category_rules = {
    "Management": lambda codes: codes.str.startswith("11-") & ~codes.str.startswith(("11-2011", "11-2021", "11-2032", "11-2033")),
    "Business and Financial Operations": lambda codes: codes.str.startswith("13-") & ~codes.str.startswith("13-1161"),
    "Computer and Information Technology": lambda codes: codes.str.startswith("15-"),
    "Architecture and Engineering": lambda codes: codes.str.startswith("17-"),
    "Life, Physical, and Social Science": lambda codes: codes.str.startswith("19-"),
    "Healthcare Practitioners and Support": lambda codes: codes.str.startswith(("29-", "31-")),
    "Education, Training, and Library": lambda codes: codes.str.startswith("25-"),
    "Arts, Design, Entertainment, Sports, and Media": lambda codes: codes.str.startswith("27-"),
    "Sales, Marketing, and Customer Service": lambda codes: codes.str.startswith(("41-", "43-4051", "11-2011", "11-2021", "11-2032", "11-2033", "13-1161")),
    "Construction and Trades": lambda codes: codes.str.startswith(("47-", "49-")),
    "Transportation and Material Moving": lambda codes: codes.str.startswith("53-"),
}

job_meta = df[["onet", "onet_name"]].drop_duplicates().set_index("onet")
job_meta["soc"] = job_meta.index

# Select 10 occupations in each family with the largest number of skills matching your lists.
# This makes the plotted proportions more informative than selecting jobs with only 1 match.
selected = {}
for category, rule in category_rules.items():
    eligible_codes = job_meta.index[rule(job_meta["soc"])]
    subset = counts[counts.index.get_level_values("onet").isin(eligible_codes)].copy()
    subset = subset[subset["total_classified"] > 0]
    subset = subset.sort_values("total_classified", ascending=False).head(10)
    if len(subset) < 10:
        raise ValueError(f"{category} has only {len(subset)} occupations with matching skills.")
    selected[category] = subset

# Check that each category has 10 occupations and no occupation is selected twice.
assert all(len(subset) == 10 for subset in selected.values())
selected_codes = [code for subset in selected.values() for code in subset.index.get_level_values("onet")]
assert len(selected_codes) == len(set(selected_codes)), "An occupation appears in more than one category."

for category, subset in selected.items():
    print(f"{category}: 10 occupations")
    print(subset.index.get_level_values("onet_name").to_list())


In [ ]:
# Create one readable horizontal 100% stacked bar chart per occupation category.
# Each chart contains 10 jobs. The three segments sum to 100% for each job.

colors = ["#4E79A7", "#E15759", "#59A14F"]

for category, subset in selected.items():
    category_counts = subset[category_order]
    percentages = category_counts.div(category_counts.sum(axis=1), axis=0) * 100

    labels = [
        f"{name} ({code}) · n={int(n)}"
        for (code, name), n in zip(subset.index.tolist(), subset["total_classified"].tolist())
    ]

    fig, ax = plt.subplots(figsize=(14, 6))
    left = np.zeros(len(percentages))

    for j, col in enumerate(category_order):
        ax.barh(
            labels,
            percentages[col].values,
            left=left,
            label=col,
            color=colors[j],
            height=0.78,
        )
        left += percentages[col].values

    ax.set_title(f"{category}: Skill Category Mix", fontsize=14, fontweight="bold")
    ax.set_xlabel("Share of matched skills within occupation (%)")
    ax.set_xlim(0, 100)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.grid(axis="x", alpha=0.25)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", labelsize=9)
    ax.invert_yaxis()
    ax.legend(title="Skill category", loc="lower right")
    plt.tight_layout()
    plt.show()

# Optional: save each category's chart as a separate PNG instead of displaying it.
# Uncomment the following lines inside the loop, before plt.show():
# safe_name = category.lower().replace(" ", "_").replace(",", "")
# fig.savefig(f"{safe_name}_skill_mix.png", dpi=180, bbox_inches="tight")


## Reading the chart

- Each bar reaches 100% because it is normalized within that occupation.
- `n=` in each label is the number of skills matching one of the three lists.
- Skills not included in the three lists are omitted from the denominator.
- The SOC family codes group occupations according to the major occupation categories in your prompt.